In [1]:
import pandas as pd
import numpy as np



print("Libraries imported successfully!")

Libraries imported successfully!


In [2]:
df = pd.read_csv("data/StudentsPerformance.csv")
df.head()

,gender,race/ethnicity,parental level of education,lunch,test preparation course,math score,reading score,writing score
0,female,group B,bachelor's degree,standard,none,72,72,74
1,female,group C,some college,standard,completed,69,90,88
2,female,group B,master's degree,standard,none,90,95,93
3,male,group A,associate's degree,free/reduced,none,47,57,44
4,male,group C,some college,standard,none,76,78,75


In [3]:
print("Dataset shape:", df.shape)

df.info()

Dataset shape: (1000, 8)
<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 8 columns):
 #   Column                       Non-Null Count  Dtype
---  ------                       --------------  -----
 0   gender                       1000 non-null   str  
 1   race/ethnicity               1000 non-null   str  
 2   parental level of education  1000 non-null   str  
 3   lunch                        1000 non-null   str  
 4   test preparation course      1000 non-null   str  
 5   math score                   1000 non-null   int64
 6   reading score                1000 non-null   int64
 7   writing score                1000 non-null   int64
dtypes: int64(3), str(5)
memory usage: 62.6 KB


In [4]:


print("Column names:")
print(df.columns.tolist())

print("\nMissing values in each column:")
print(df.isnull().sum())

print("\nNumber of duplicate rows:", df.duplicated().sum())


print("\nData types:")
print(df.dtypes)

print("\nNumerical summary:")
print(df.describe())


Column names:
['gender', 'race/ethnicity', 'parental level of education', 'lunch', 'test preparation course', 'math score', 'reading score', 'writing score']

Missing values in each column:
gender                         0
race/ethnicity                 0
parental level of education    0
lunch                          0
test preparation course        0
math score                     0
reading score                  0
writing score                  0
dtype: int64

Number of duplicate rows: 0

Data types:
gender                           str
race/ethnicity                   str
parental level of education      str
lunch                            str
test preparation course          str
math score                     int64
reading score                  int64
writing score                  int64
dtype: object

Numerical summary:
       math score  reading score  writing score
count  1000.00000    1000.000000    1000.000000
mean     66.08900      69.169000      68.054000
std      15.16308

In [5]:

def clean_data(data):
    
    cleaned = data.copy()

   
    cleaned.columns = (
        cleaned.columns.str.strip()
        .str.lower()
        .str.replace(r"\s+", "_", regex=True)
        .str.replace("/", "_", regex=False)
    )

    
    for column in cleaned.select_dtypes(include=["str", "object"]).columns:
        cleaned[column] = cleaned[column].str.strip()

    
    cleaned = cleaned.drop_duplicates()

    
    score_columns = [
        "math_score",
        "reading_score",
        "writing_score"
    ]

    for column in score_columns:
        cleaned[column] = pd.to_numeric(
            cleaned[column], errors="coerce"
        )

        
        cleaned.loc[
            ~cleaned[column].between(0, 100),
            column
        ] = np.nan

    cleaned = cleaned.dropna(subset=score_columns)

    
    for column in cleaned.select_dtypes(include=["str", "object"]).columns:
        cleaned[column] = cleaned[column].fillna("Unknown")

    return cleaned.reset_index(drop=True)


In [6]:

cleaned_df = clean_data(df)

print("Original dataset shape:", df.shape)
print("Cleaned dataset shape:", cleaned_df.shape)

print("\nMissing values after cleaning:")
print(cleaned_df.isnull().sum())

print("\nDuplicate rows after cleaning:")
print(cleaned_df.duplicated().sum())

cleaned_df.head()


Original dataset shape: (1000, 8)
Cleaned dataset shape: (1000, 8)

Missing values after cleaning:
gender                         0
race_ethnicity                 0
parental_level_of_education    0
lunch                          0
test_preparation_course        0
math_score                     0
reading_score                  0
writing_score                  0
dtype: int64

Duplicate rows after cleaning:
0


,gender,race_ethnicity,parental_level_of_education,lunch,test_preparation_course,math_score,reading_score,writing_score
0,female,group B,bachelor's degree,standard,none,72.0,72.0,74.0
1,female,group C,some college,standard,completed,69.0,90.0,88.0
2,female,group B,master's degree,standard,none,90.0,95.0,93.0
3,male,group A,associate's degree,free/reduced,none,47.0,57.0,44.0
4,male,group C,some college,standard,none,76.0,78.0,75.0


In [7]:


scores = cleaned_df[
    ["math_score", "reading_score", "writing_score"]
].to_numpy()


subjects = ["Math", "Reading", "Writing"]

for i, subject in enumerate(subjects):
    subject_scores = scores[:, i]

    print(f"\n{subject} Statistics")
    print("Average:", round(np.mean(subject_scores), 2))
    print("Median:", round(np.median(subject_scores), 2))
    print("Standard deviation:", round(np.std(subject_scores), 2))
    print("Minimum score:", np.min(subject_scores))
    print("Maximum score:", np.max(subject_scores))



Math Statistics
Average: 66.09
Median: 66.0
Standard deviation: 15.16
Minimum score: 0.0
Maximum score: 100.0

Reading Statistics
Average: 69.17
Median: 70.0
Standard deviation: 14.59
Minimum score: 17.0
Maximum score: 100.0

Writing Statistics
Average: 68.05
Median: 69.0
Standard deviation: 15.19
Minimum score: 10.0
Maximum score: 100.0


In [8]:


cleaned_df["average_score"] = np.mean(scores, axis=1)

print("First 10 student averages:")
print(cleaned_df["average_score"].head(10))

print("\nOverall class average:",
      round(np.mean(cleaned_df["average_score"]), 2))


First 10 student averages:
0    72.666667
1    82.333333
2    92.666667
3    49.333333
4    76.333333
5    77.333333
6    91.666667
7    40.666667
8    65.000000
9    49.333333
Name: average_score, dtype: float64

Overall class average: 67.77


In [9]:


test_data = pd.DataFrame({
    "gender": [" female ", " female ", "male", "female"],
    "race/ethnicity": ["group B", "group B", "group A", "group C"],
    "parental level of education": [
        "some college", "some college", "high school", "bachelor's degree"
    ],
    "lunch": ["standard", "standard", "standard", "standard"],
    "test preparation course": ["none", "none", "completed", "none"],
    "math score": [80, 80, 120, np.nan],
    "reading score": [75, 75, 65, 70],
    "writing score": [78, 78, 60, 72]
})

test_cleaned = clean_data(test_data)

print("Rows before cleaning:", len(test_data))
print("Rows after cleaning:", len(test_cleaned))
print("\nCleaned test data:")
display(test_cleaned)


Rows before cleaning: 4
Rows after cleaning: 1

Cleaned test data:


,gender,race_ethnicity,parental_level_of_education,lunch,test_preparation_course,math_score,reading_score,writing_score
0,female,group B,some college,standard,none,80.0,75.0,78.0


In [10]:
test_data = pd.DataFrame({
    "gender": [" female ", " female ", "male", "female"],
    "race/ethnicity": ["group B", "group B", "group A", "group C"],
    "parental level of education": [
        "some college", "some college",
        "high school", "bachelor's degree"
    ],
    "lunch": ["standard", "standard", "standard", "standard"],
    "test preparation course": ["none", "none", "completed", "none"],
    "math score": [80, 80, 120, np.nan],
    "reading score": [75, 75, 65, 70],
    "writing score": [78, 78, 60, 72]
})

test_cleaned = clean_data(test_data)

print("Rows before cleaning:", len(test_data))
print("Rows after cleaning:", len(test_cleaned))

display(test_cleaned)

Rows before cleaning: 4
Rows after cleaning: 1


,gender,race_ethnicity,parental_level_of_education,lunch,test_preparation_course,math_score,reading_score,writing_score
0,female,group B,some college,standard,none,80.0,75.0,78.0


In [11]:

scores = cleaned_df[
    ["math_score", "reading_score", "writing_score"]
].to_numpy()

print("25th percentile:", np.percentile(scores, 25, axis=0))
print("75th percentile:", np.percentile(scores, 75, axis=0))

25th percentile: [57.   59.   57.75]
75th percentile: [77. 79. 79.]


In [12]:
# Calculate student averages
cleaned_df["average_score"] = np.mean(scores, axis=1)

cleaned_df["performance"] = np.where(
    cleaned_df["average_score"] >= 50,
    "Pass",
    "Fail"
)

cleaned_df[
    ["math_score", "reading_score", "writing_score",
     "average_score", "performance"]
].head(10)

,math_score,reading_score,writing_score,average_score,performance
0,72.0,72.0,74.0,72.666667,Pass
1,69.0,90.0,88.0,82.333333,Pass
2,90.0,95.0,93.0,92.666667,Pass
3,47.0,57.0,44.0,49.333333,Fail
4,76.0,78.0,75.0,76.333333,Pass
5,71.0,83.0,78.0,77.333333,Pass
6,88.0,95.0,92.0,91.666667,Pass
7,40.0,43.0,39.0,40.666667,Fail
8,64.0,64.0,67.0,65.000000,Pass
9,38.0,60.0,50.0,49.333333,Fail


## Conclusion

This project demonstrates how Python can be used to load, clean, and analyze student performance data. Pandas was used to handle missing values, duplicates, and data formatting, while NumPy was used to calculate statistical measures such as mean, median, standard deviation, and percentiles.

The analysis shows that students performed highest in reading, followed by writing and mathematics. The overall average score was approximately 67.77.